### 1. Import Library

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
import gdown
import zipfile
import os
import shutil
import random

### 2. Persiapan Dataset

Download dataset dari Google Drive

In [ ]:
file_id = '1o0b6Nqs89zYoyRcnih5oGS7sk0bIrImo'
url = f'https://drive.google.com/uc?id={file_id}'
output_zip = 'dataset.zip'

print("Mengunduh dataset dari Google Drive...")
gdown.download(url, output_zip, quiet=False)

Extract Dataset

In [ ]:
extract_dir = 'dataset'
print("\nMengekstrak dataset...")
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

### 2.b. Analisis Imbalance Data (dan Persiapan PINN)
Ground truth mask memiliki masalah imbalance kelas yang ekstrem (lesi vs background). Analisis ini penting karena **Physics-Informed Neural Network (PINN)** dengan Total Variation (TV) loss akan membantu mengurangi efek imbalance dengan memaksa kontur segmentasi yang halus dan kontinu secara fisika.

In [ ]:
import os
import glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def analyze_class_imbalance(base_dir):
    mask_dir = os.path.join(base_dir, 'train', 'masks')
    mask_files = glob.glob(os.path.join(mask_dir, '*'))
    
    total_pixels = 0
    total_white_pixels = 0
    
    print(f"Menganalisis {len(mask_files)} mask pada dataset train...")
    for mask_path in mask_files:
        img = Image.open(mask_path).convert('L')
        arr = np.array(img)
        binary = (arr > 127).astype(np.uint8) if arr.max() > 1 else (arr > 0.5).astype(np.uint8)
        total_pixels += binary.size
        total_white_pixels += np.sum(binary)
        
    white_percent = (total_white_pixels / total_pixels) * 100
    bg_percent = 100.0 - white_percent
    
    print(f"\n--- Analisis Imbalance Data ---")
    print(f"Total Piksel: {total_pixels:,}")
    print(f"Piksel Lesi (White): {total_white_pixels:,} ({white_percent:.4f}%)")
    print(f"Piksel Background: {total_pixels - total_white_pixels:,} ({bg_percent:.4f}%)")
    print(f"\n>>> Rasio Background:Lesi = {bg_percent/white_percent:.1f}:1 <<<")
    print(f">>> Ini adalah imbalance EKSTREM yang memerlukan penanganan khusus <<<")
    
    # Visualisasi
    classes = ['Background', 'Lesion (Stroke)']
    percentages = [bg_percent, white_percent]
    
    plt.figure(figsize=(6, 4))
    bars = plt.bar(classes, percentages, color=['#34495e', '#e74c3c'])
    plt.ylabel('Persentase (%)')
    plt.title('Distribusi Kelas pada Ground Truth Mask (Piksel)')
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')
    plt.ylim(0, 110)
    plt.show()

analyze_class_imbalance('dataset')

### 3. NCCTDataset

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF
from PIL import Image
import random
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import segmentation_models_pytorch as smp

class NCCTDataset(Dataset):
    def __init__(self, split_dir, limit_size=None, transform=None):
        self.split_dir = split_dir
        self.transform = transform

        img_dir = os.path.join(split_dir, 'images')
        mask_dir = os.path.join(split_dir, 'masks')

        self.image_paths = sorted([os.path.join(img_dir, f) for f in os.listdir(img_dir)])
        self.mask_paths = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir)])

        if limit_size is not None:
            self.image_paths = self.image_paths[:limit_size]
            self.mask_paths = self.mask_paths[:limit_size]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_pil = Image.open(self.image_paths[idx]).convert("L")
        mask_pil = Image.open(self.mask_paths[idx]).convert("L")

        if self.transform:
            if hasattr(self.transform, 'processors') or 'Compose' in str(type(self.transform)):
                # Albumentations
                img_np = np.array(img_pil)
                mask_np = np.array(mask_pil)
                augmented = self.transform(image=img_np, mask=mask_np)
                img = augmented['image']
                mask = augmented['mask']
            else:
                # Torchvision fallback
                img = img_pil
                mask = mask_pil
                if self.split_dir.endswith('train'):
                    if random.random() > 0.5:
                        img = TF.hflip(img)
                        mask = TF.hflip(mask)
                    if random.random() > 0.5:
                        img = TF.vflip(img)
                        mask = TF.vflip(mask)
                    angle = random.uniform(-15, 15)
                    img = TF.rotate(img, angle)
                    mask = TF.rotate(mask, angle)
                img = self.transform(img)
                mask = self.transform(mask)
        else:
            img = TF.to_tensor(img_pil)
            mask = TF.to_tensor(mask_pil)

        # Binarize mask to 0.0 or 1.0 (Fixes bilinear interpolation bug)
        if isinstance(mask, torch.Tensor):
            mask = mask.float()
            if mask.max() > 1.0:
                mask = (mask > 127.0).float()
            else:
                mask = (mask > 0.5).float()
            if mask.ndim == 2:
                mask = mask.unsqueeze(0)
        else:
            mask_np = np.array(mask)
            if mask_np.max() > 1.0:
                mask = torch.tensor((mask_np > 127.0).astype(np.float32)).unsqueeze(0)
            else:
                mask = torch.tensor((mask_np > 0.5).astype(np.float32)).unsqueeze(0)

        label = 1.0 if torch.max(mask) > 0 else 0.0

        return img, mask, torch.tensor([label], dtype=torch.float32)

Cek Dataset

In [ ]:
import matplotlib.pyplot as plt

def get_dataloaders(base_dir, batch_size=4, limit_size=None):
    train_transform = A.Compose([
        A.Resize(256, 256),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2(),
    ])

    val_transform = A.Compose([
        A.Resize(256, 256),
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2(),
    ])

    train_dataset = NCCTDataset(os.path.join(base_dir, 'train'), limit_size=limit_size, transform=train_transform)
    val_dataset = NCCTDataset(os.path.join(base_dir, 'val'), limit_size=limit_size, transform=val_transform)
    test_dataset = NCCTDataset(os.path.join(base_dir, 'test'), limit_size=limit_size, transform=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

# Ambil 1 batch data untuk verifikasi
train_loader, _, _ = get_dataloaders('dataset', batch_size=4)
images, masks, labels = next(iter(train_loader))

print(f"Image shape: {images.shape}")
print(f"Mask shape: {masks.shape}")
print(f"Max mask value: {masks.max():.2f}")
print(f"Min mask value: {masks.min():.2f}")

# Visualisasi sederhana
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(images[0][0].cpu().numpy(), cmap='gray'); plt.title("Input Image")
plt.subplot(1, 2, 2); plt.imshow(masks[0][0].cpu().numpy(), cmap='gray'); plt.title("Ground Truth Mask")
plt.show()

### 4. Model Arsitektur

#### 4.a. CSPNet

In [ ]:
class CSPBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        mid_channels = out_channels // 2

        self.main_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True)
        )

        self.skip_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True)
        )

        self.transition = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        main_out = self.main_conv(x)
        skip_out = self.skip_conv(x)
        merged = torch.cat([main_out, skip_out], dim=1)
        return self.transition(merged)

class NCCTSegmentationModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder (Backbone)
        self.initial_conv = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.layer1 = CSPBlock(32, 64)   # Output: 64 channels
        self.layer2 = CSPBlock(64, 128)  # Output: 128 channels
        self.layer3 = CSPBlock(128, 256) # Output: 256 channels

        self.pool = nn.MaxPool2d(2, 2)

        # Decoder with Skip Connections
        # features (256 channels) -> upconv1 (128 channels)
        self.upconv1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        # concat(up1:128, layer3:256) = 384
        self.decoder1 = CSPBlock(128 + 256, 128)

        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        # concat(up2:64, layer2:128) = 192
        self.decoder2 = CSPBlock(64 + 128, 64)

        self.upconv3 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        # concat(up3:32, layer1:64) = 96
        self.decoder3 = CSPBlock(32 + 64, 32)

        self.seg_output = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        # Encoder
        x0 = F.relu(self.initial_conv(x))

        x1 = self.layer1(x0)
        p1 = self.pool(x1)

        x2 = self.layer2(p1)
        p2 = self.pool(x2)

        x3 = self.layer3(p2)
        features = self.pool(x3)

        # Decoder
        d1 = self.upconv1(features)
        d1 = torch.cat([d1, x3], dim=1)
        d1 = self.decoder1(d1)

        d2 = self.upconv2(d1)
        d2 = torch.cat([d2, x2], dim=1)
        d2 = self.decoder2(d2)

        d3 = self.upconv3(d2)
        d3 = torch.cat([d3, x1], dim=1)
        d3 = self.decoder3(d3)

        seg_out = self.seg_output(d3)
        return seg_out

#### 4.b. UNet (ResNet34)

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.model = smp.Unet(
            encoder_name="resnet34",
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=out_channels,
            activation=None
        )

    def forward(self, x):
        return self.model(x)


#### 4.c. UNet EfficientNet-b3

In [ ]:
class UNet_EffNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.model = smp.Unet(
            encoder_name="efficientnet-b3",
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=out_channels,
            activation=None
        )
    def forward(self, x):
        return self.model(x)


#### 4.d. FPN (ResNeXt50)

In [ ]:
class FPN_ResNext(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.model = smp.FPN(
            encoder_name="resnext50_32x4d",
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=out_channels,
            activation=None
        )
    def forward(self, x):
        return self.model(x)


---
### 5. PINN (Physics-Informed Neural Network) - Total Variation Loss for Imbalance Class

**Apa itu PINN?**

Physics-Informed Neural Networks (PINNs) adalah framework yang mengintegrasikan hukum fisika ke dalam proses training jaringan saraf. Dalam konteks segmentasi citra medis, **PINN memanfaatkan prior pengetahuan fisika untuk menghasilkan segmentasi yang lebih realistis dan robust terhadap class imbalance.**

**Mengapa PINN membantu class imbalance?**

Pada NCCT stroke, lesi biasanya:
- Berukuran sangat kecil (<1% piksel) → class imbalance ekstrem
- Berbentuk kontinu (tidak terpecah-pecah) → properti fisik
- Memiliki batas yang halus → properti fisik

PINN menambahkan **Total Variation (TV) Loss** sebagai regularisasi fisika yang:
1. **Meminimalkan gradien spasial** pada prediksi → mendorong segmentasi yang halus
2. **Mengurangi noise** dan prediksi salt-and-pepper yang sering muncul akibat class imbalance
3. **Memaksa kontur lesi yang kontinu** secara spasial → lebih realistis secara anatomis

**Rumus TV Loss:**

$$\mathcal{L}_{TV} = \frac{1}{N} \sum_{i,j} \left( |p_{i+1,j} - p_{i,j}| + |p_{i,j+1} - p_{i,j}| \right)$$

Dimana $p$ adalah prediksi probabilitas (setelah sigmoid), dan $(i,j)$ adalah koordinat spasial.

**Loss Total untuk V7:**

$$\mathcal{L}_{total} = \lambda_1 \cdot \mathcal{L}_{WFocal} + \lambda_2 \cdot \mathcal{L}_{Tversky} + \lambda_3 \cdot \mathcal{L}_{TV}$$

Referensi utama: *PINN-EMFNet: PINN-based and Enhanced Multi-Scale Feature Fusion Network for Breast Ultrasound Images Segmentation* (Ding et al., 2024) - Menggunakan kombinasi BCE + Soft Dice + TV Loss.

In [ ]:
class TotalVariationLoss(nn.Module):
    """
    Physics-Informed Total Variation (TV) Loss.
    
    TV loss menghitung variasi total dari prediksi probabilitas,
    yang merepresentasikan prior fisika bahwa segmentasi harus halus
    dan kontinu secara spasial.
    
    Rumus: TV(p) = (1/N) * sum(|p(i+1,j) - p(i,j)| + |p(i,j+1) - p(i,j)|)
    
    Parameter:
        beta: Tingkat smoothing (lower = lebih smooth). Default: 1.0
    """
    def __init__(self, beta=1.0):
        super(TotalVariationLoss, self).__init__()
        self.beta = beta
    
    def forward(self, logits):
        # Apply sigmoid to get probabilities
        probs = torch.sigmoid(logits)
        
        # Calculate spatial gradients (differences between adjacent pixels)
        # Horizontal gradient: |p(i,j+1) - p(i,j)|
        diff_h = torch.abs(probs[:, :, :, 1:] - probs[:, :, :, :-1])
        
        # Vertical gradient: |p(i+1,j) - p(i,j)|
        diff_v = torch.abs(probs[:, :, 1:, :] - probs[:, :, :-1, :])
        
        # Total Variation = mean of all gradient magnitudes
        tv_loss = (torch.mean(diff_h) + torch.mean(diff_v)) / 2.0
        
        return tv_loss


class PINN_BCEDiceTverskyTVLoss(nn.Module):
    """
    Combined loss function with Physics-Informed Total Variation regularization.
    
    Menggabungkan:
    1. Weighted Focal Loss (mengatasi class imbalance di level piksel)
    2. Tversky Loss (mengatasi class imbalance di level region, fokus pada false negatives)
    3. Total Variation Loss (PINN - physics-informed smoothness constraint)
    
    Parameter:
        bce_weight: Bobot untuk focal loss vs tversky loss (default: 0.5)
        gamma: Focal loss focusing parameter (default: 2.0)
        pos_weight: Bobot untuk kelas positif (lesi) - mengatasi imbalance
        tv_weight: Bobot untuk Total Variation physics loss (default: 0.01)
        alpha: Tversky alpha (penalti FP) (default: 0.3)
        beta: Tversky beta (penalti FN) (default: 0.7) -> lebih fokus pada false negatives
    """
    def __init__(self, bce_weight=0.5, gamma=2.0, pos_weight=66.0, 
                 tv_weight=0.01, alpha=0.3, beta=0.7):
        super(PINN_BCEDiceTverskyTVLoss, self).__init__()
        self.bce_weight = bce_weight
        self.gamma = gamma
        self.pos_weight = pos_weight
        self.tv_weight = tv_weight
        self.alpha = alpha
        self.beta = beta
        self.tv_loss_fn = TotalVariationLoss()

    def forward(self, inputs, targets):
        # 1. Weighted Focal Loss (pixel-level imbalance)
        pw = torch.tensor([self.pos_weight], device=inputs.device)
        bce = F.binary_cross_entropy_with_logits(inputs, targets, pos_weight=pw, reduction='none')
        pt = torch.exp(-bce)
        focal_loss = ((1 - pt) ** self.gamma * bce).mean()

        # 2. Tversky Loss (region-level imbalance, penalizes FN more)
        inputs_sigmoid = torch.sigmoid(inputs)
        intersection = (inputs_sigmoid * targets).sum(dim=(2, 3))
        fps = (inputs_sigmoid * (1.0 - targets)).sum(dim=(2, 3))
        fns = ((1.0 - inputs_sigmoid) * targets).sum(dim=(2, 3))
        
        tversky_score = (intersection + 1e-6) / (intersection + self.alpha * fps + self.beta * fns + 1e-6)
        tversky_loss = 1.0 - tversky_score.mean()

        # 3. PINN: Total Variation Loss (physics-informed smoothness constraint)
        tv_loss = self.tv_loss_fn(inputs)

        # Combine losses
        combined = (self.bce_weight * focal_loss) + ((1 - self.bce_weight) * tversky_loss)
        total_loss = combined + self.tv_weight * tv_loss
        
        return total_loss


class BCEDiceTverskyLoss(nn.Module):
    """Same as above but WITHOUT TV Loss (for ablation study)"""
    def __init__(self, bce_weight=0.5, gamma=2.0, pos_weight=66.0, alpha=0.3, beta=0.7):
        super(BCEDiceTverskyLoss, self).__init__()
        self.bce_weight = bce_weight
        self.gamma = gamma
        self.pos_weight = pos_weight
        self.alpha = alpha
        self.beta = beta

    def forward(self, inputs, targets):
        pw = torch.tensor([self.pos_weight], device=inputs.device)
        bce = F.binary_cross_entropy_with_logits(inputs, targets, pos_weight=pw, reduction='none')
        pt = torch.exp(-bce)
        focal_loss = ((1 - pt) ** self.gamma * bce).mean()

        inputs_sigmoid = torch.sigmoid(inputs)
        intersection = (inputs_sigmoid * targets).sum(dim=(2, 3))
        fps = (inputs_sigmoid * (1.0 - targets)).sum(dim=(2, 3))
        fns = ((1.0 - inputs_sigmoid) * targets).sum(dim=(2, 3))
        
        tversky_score = (intersection + 1e-6) / (intersection + self.alpha * fps + self.beta * fns + 1e-6)
        tversky_loss = 1.0 - tversky_score.mean()

        return (self.bce_weight * focal_loss) + ((1 - self.bce_weight) * tversky_loss)


def dice_coeff(pred, target, threshold=0.5, epsilon=1e-6):
    pred = (torch.sigmoid(pred) > threshold).float()
    target = (target > 0.5).float()
    inter = (pred * target).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    dice = (2. * inter + epsilon) / (union + epsilon)
    return dice.mean().item()

### 6. Visualisasi Efek TV Loss

Cell ini mendemonstrasikan bagaimana **Total Variation Loss** bekerja sebagai regularisasi fisika. TV loss menghitung perbedaan intensitas antar piksel tetangga. Semakin halus prediksi, semakin kecil TV loss.

In [ ]:
# Demonstrasi efek TV Loss
tv_loss_demo = TotalVariationLoss()

# Buat 3 skenario: smooth, moderate, noisy prediction
smooth_pred = torch.sigmoid(torch.full((1, 1, 50, 50), 2.0))  # Seragam
noisy_pred = torch.sigmoid(torch.randn(1, 1, 50, 50) * 3.0)  # Random noise
edge_pred = torch.sigmoid(torch.ones(1, 1, 50, 50) * 2.0)  # With sharp edge
edge_pred[:, :, :, 25:] = torch.sigmoid(torch.tensor(-2.0))

print("Demonstrasi Physics-Informed Total Variation Loss:")
print(f"  Smooth prediction TV loss: {tv_loss_demo(smooth_pred):.6f}")
print(f"  Noisy prediction TV loss:  {tv_loss_demo(noisy_pred):.6f}  (HIGH - akan dipenalty)")
print(f"  Edge prediction TV loss:   {tv_loss_demo(edge_pred):.6f}")
print()
print("=> TV loss memberikan penalty lebih tinggi pada prediksi yang tidak halus,")
print("   memaksa model menghasilkan segmentasi yang kontinu dan realistis secara fisika.")

# Visualisasi
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(smooth_pred[0,0].numpy(), cmap='hot', vmin=0, vmax=1)
axes[0].set_title(f"Smooth\nTV Loss: {tv_loss_demo(smooth_pred):.4f}")
axes[1].imshow(noisy_pred[0,0].numpy(), cmap='hot', vmin=0, vmax=1)
axes[1].set_title(f"Noisy (salt-and-pepper)\nTV Loss: {tv_loss_demo(noisy_pred):.4f}")
axes[2].imshow(edge_pred[0,0].numpy(), cmap='hot', vmin=0, vmax=1)
axes[2].set_title(f"With Edge\nTV Loss: {tv_loss_demo(edge_pred):.4f}")
plt.tight_layout()
plt.show()

### 7. Fungsi Training dan Evaluasi

Fungsi training sekarang mendukung **ablation mode** - bisa training dengan atau tanpa PINN TV Loss untuk perbandingan.

In [ ]:
import time

def train_model(model, train_loader, val_loader, epochs=40, model_name="Model", 
                patience=10, min_delta=0.001, use_pinn=True):
    """
    Train model dengan opsi PINN (Physics-Informed) TV Loss.
    
    Args:
        use_pinn: Jika True, gunakan PINN_BCEDiceTverskyTVLoss (dengan TV Loss).
                  Jika False, gunakan BCEDiceTverskyLoss (tanpa TV Loss) untuk ablation.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    if use_pinn:
        criterion = PINN_BCEDiceTverskyTVLoss(bce_weight=0.5, tv_weight=0.01).cuda()
        loss_tag = "PINN-TV"
    else:
        criterion = BCEDiceTverskyLoss(bce_weight=0.5).cuda()
        loss_tag = "No-PINN"

    print(f"\nTraining {model_name} [Loss: {loss_tag}]...")
    print(f"{'Epoch':<8} | {'T-Loss':<8} | {'V-Loss':<8} | {'V-Dice':<8} | {'LR':<10} | {'Time':<8}")
    print("-" * 70)

    best_dice = 0
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'val_dice': []}

    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        train_loss = 0
        for images, masks, _ in train_loader:
            images, masks = images.cuda(), masks.cuda()
            optimizer.zero_grad()
            seg_out = model(images)
            loss = criterion(seg_out, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss, val_dice = 0, 0
        with torch.no_grad():
            for images, masks, _ in val_loader:
                images, masks = images.cuda(), masks.cuda()
                seg_out = model(images)
                loss = criterion(seg_out, masks)
                val_loss += loss.item()
                val_dice += dice_coeff(seg_out, masks)

        end_time = time.time()
        duration = end_time - start_time

        avg_t_loss = train_loss / len(train_loader)
        avg_v_loss = val_loss / len(val_loader)
        avg_v_dice = val_dice / len(val_loader)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(avg_t_loss)
        history['val_loss'].append(avg_v_loss)
        history['val_dice'].append(avg_v_dice)

        scheduler.step(avg_v_loss)

        print(f"{epoch+1:<8} | {avg_t_loss:<8.4f} | {avg_v_loss:<8.4f} | {avg_v_dice:<8.4f} | {current_lr:<10.6f} | {duration:<8.2f}s")

        if avg_v_dice > best_dice:
            best_dice = avg_v_dice
            tag = "pinn" if use_pinn else "nopinn"
            torch.save(model.state_dict(), f"{model_name.lower().replace(' ', '_')}_{tag}_best.pth")

        if avg_v_loss < best_val_loss - min_delta:
            best_val_loss = avg_v_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break

    return model, history


def eval_model(models_dict, test_loader):
    results = {}
    print(f"\n{'Model':<20} | {'Test Dice Score':<15}")
    print("-" * 38)

    for name, model in models_dict.items():
        model.eval()
        test_dice = 0
        with torch.no_grad():
            for images, masks, _ in test_loader:
                images, masks = images.cuda(), masks.cuda()
                seg_out = model(images)
                test_dice += dice_coeff(seg_out, masks)

        final_dice = test_dice / len(test_loader)
        results[name] = final_dice
        print(f"{name:<20} | {final_dice:<15.4f}")

    return results

---
### 8. Training & Ablation Study: PINN vs Non-PINN

Kita akan melakukan **ablation study** untuk membandingkan performa model dengan dan tanpa PINN TV Loss.

**Hipotesis:**
- Model dengan PINN TV Loss akan menghasilkan segmentasi yang lebih halus (lebih sedikit noise)
- Model dengan PINN akan memiliki Dice Score yang lebih tinggi pada data testing
- PINN akan sangat membantu pada kasus class imbalance ekstrem dengan mengurangi false positives

Inisialisasi DataLoader

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders('dataset')
trained_models_pinn = {}  # Model dengan PINN
trained_models_nopinn = {}  # Model tanpa PINN (ablation)
model_histories = {}

#### 8.a. UNet (ResNet34) - Perbandingan PINN vs Non-PINN

In [ ]:
# --- UNet dengan PINN ---
name = "UNet_ResNet34"
model_pinn = UNet().cuda()
m_pinn, h_pinn = train_model(model_pinn, train_loader, val_loader, epochs=40, 
                               model_name=f"{name}_PINN", use_pinn=True)
trained_models_pinn[name] = m_pinn
model_histories[f"{name}_PINN"] = h_pinn

In [ ]:
# --- UNet tanpa PINN (ablation) ---
model_nopinn = UNet().cuda()
m_nopinn, h_nopinn = train_model(model_nopinn, train_loader, val_loader, epochs=40, 
                                   model_name=f"{name}_NoPINN", use_pinn=False)
trained_models_nopinn[name] = m_nopinn
model_histories[f"{name}_NoPINN"] = h_nopinn

#### 8.b. UNet EfficientNet-b3 - Perbandingan PINN vs Non-PINN

In [ ]:
# --- UNet EffNet dengan PINN ---
name = "UNet_EffNet"
model_pinn = UNet_EffNet().cuda()
m_pinn, h_pinn = train_model(model_pinn, train_loader, val_loader, epochs=40, 
                               model_name=f"{name}_PINN", use_pinn=True)
trained_models_pinn[name] = m_pinn
model_histories[f"{name}_PINN"] = h_pinn

In [ ]:
# --- UNet EffNet tanpa PINN (ablation) ---
model_nopinn = UNet_EffNet().cuda()
m_nopinn, h_nopinn = train_model(model_nopinn, train_loader, val_loader, epochs=40, 
                                   model_name=f"{name}_NoPINN", use_pinn=False)
trained_models_nopinn[name] = m_nopinn
model_histories[f"{name}_NoPINN"] = h_nopinn

#### 8.c. FPN ResNeXt50 - Perbandingan PINN vs Non-PINN

In [ ]:
# --- FPN dengan PINN ---
name = "FPN_ResNeXt"
model_pinn = FPN_ResNext().cuda()
m_pinn, h_pinn = train_model(model_pinn, train_loader, val_loader, epochs=40, 
                               model_name=f"{name}_PINN", use_pinn=True)
trained_models_pinn[name] = m_pinn
model_histories[f"{name}_PINN"] = h_pinn

In [ ]:
# --- FPN tanpa PINN (ablation) ---
model_nopinn = FPN_ResNext().cuda()
m_nopinn, h_nopinn = train_model(model_nopinn, train_loader, val_loader, epochs=40, 
                                   model_name=f"{name}_NoPINN", use_pinn=False)
trained_models_nopinn[name] = m_nopinn
model_histories[f"{name}_NoPINN"] = h_nopinn

### 9. Evaluasi Ablation Study: PINN vs Non-PINN

In [ ]:
print("=" * 60)
print("           ABLATION STUDY: PINN vs Non-PINN")
print("=" * 60)

print("\n--- Model dengan PINN (Physics-Informed TV Loss) ---")
results_pinn = eval_model(trained_models_pinn, test_loader)

print("\n--- Model tanpa PINN (Baseline) ---")
results_nopinn = eval_model(trained_models_nopinn, test_loader)

print("\n" + "=" * 60)
print("           PERBANDINGAN HASIL")
print("=" * 60)
print(f"\n{'Model':<20} | {'PINN Dice':<10} | {'No-PINN Dice':<13} | {'Gain':<8}")
print("-" * 55)

total_gain = 0
count = 0
for model_name in results_pinn:
    pinn_score = results_pinn[model_name] * 100
    nopinn_score = results_nopinn[model_name] * 100
    gain = pinn_score - nopinn_score
    gain_str = f"+{gain:.2f}" if gain > 0 else f"{gain:.2f}"
    print(f"{model_name:<20} | {pinn_score:<8.2f}% | {nopinn_score:<11.2f}% | {gain_str:<8}")
    total_gain += gain
    count += 1

if count > 0:
    print("-" * 55)
    print(f"{'Average':<20} | {'':8} | {'':11} | {total_gain/count:+.2f}")
    
print("\n" + "=" * 60)
if total_gain > 0:
    print("  >>> PINN TV Loss MENINGKATKAN performa segmentasi! <<<")
else:
    print("  >>> PINN TV Loss memberikan efek netral/menurun. <<<")
    print("  >>> Catatan: Hiperparameter tv_weight mungkin perlu tuning. <<<")
print("=" * 60)

### 10. Visualisasi Perbandingan PINN vs Non-PINN

Visualisasi langsung menunjukkan perbedaan kualitas segmentasi. Model dengan PINN diharapkan memiliki segmentasi yang lebih halus dengan lebih sedikit false positive.

In [ ]:
def visualize_ablation(pinn_models, nopinn_models, test_loader, num_samples=3):
    model_names = list(pinn_models.keys())
    num_models = len(model_names)
    
    # Each sample: Input | GT | ModelA_NoPINN | ModelA_PINN | ModelB_NoPINN | ModelB_PINN | ...
    cols_per_model = 2  # NoPINN and PINN
    total_cols = 2 + num_models * cols_per_model  # 2 (Input + GT) + 2 per model
    
    plt.figure(figsize=(total_cols * 3, num_samples * 3.5))

    with torch.no_grad():
        for i in range(num_samples):
            images, masks, _ = next(iter(test_loader))
            images_gpu = images.cuda()

            # Row i - Plot Input
            col = 1
            plt.subplot(num_samples, total_cols, i * total_cols + col)
            plt.imshow(images[0][0].cpu().numpy(), cmap='gray')
            plt.title("Input")
            plt.axis('off')

            # Plot Ground Truth
            col += 1
            plt.subplot(num_samples, total_cols, i * total_cols + col)
            gt = masks[0][0].cpu().numpy()
            plt.imshow(gt, cmap='gray')
            plt.title("GT")
            plt.axis('off')

            # Plot each model's predictions (NoPINN vs PINN side by side)
            for j, name in enumerate(model_names):
                # NoPINN
                col += 1
                model_np = nopinn_models[name]
                model_np.eval()
                out_np = model_np(images_gpu)
                pred_np = (torch.sigmoid(out_np) > 0.5).cpu().numpy()[0][0]
                
                plt.subplot(num_samples, total_cols, i * total_cols + col)
                plt.imshow(pred_np, cmap='gray')
                plt.title(f"{name}\nNo-PINN")
                plt.axis('off')

                # PINN
                col += 1
                model_p = pinn_models[name]
                model_p.eval()
                out_p = model_p(images_gpu)
                pred_p = (torch.sigmoid(out_p) > 0.5).cpu().numpy()[0][0]
                
                plt.subplot(num_samples, total_cols, i * total_cols + col)
                plt.imshow(pred_p, cmap='gray')
                plt.title(f"{name}\nPINN-TV")
                plt.axis('off')

    plt.tight_layout()
    plt.show()

# Visualisasi perbandingan
if trained_models_pinn and trained_models_nopinn:
    visualize_ablation(trained_models_pinn, trained_models_nopinn, test_loader, num_samples=3)
else:
    print("Model belum dilatih. Jalankan sel training terlebih dahulu.")

### 11. Analisis Training Curves

Bandingkan kurva training dan validation loss antara model dengan PINN dan tanpa PINN.

In [ ]:
def plot_ablation_curves(histories):
    """Plot training curves untuk perbandingan PINN vs No-PINN."""
    # Filter histories by PINN vs NoPINN
    pinn_keys = [k for k in histories.keys() if 'PINN' in k and 'NoPINN' not in k]
    nopinn_keys = [k for k in histories.keys() if 'NoPINN' in k]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot train loss
    ax = axes[0, 0]
    for k in pinn_keys:
        ax.plot(histories[k]['train_loss'], '--', label=f"{k} (PINN)", linewidth=2)
    for k in nopinn_keys:
        ax.plot(histories[k]['train_loss'], ':', label=f"{k}", linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Train Loss')
    ax.set_title('Training Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot val loss
    ax = axes[0, 1]
    for k in pinn_keys:
        ax.plot(histories[k]['val_loss'], '-', label=f"{k} (PINN)", linewidth=2)
    for k in nopinn_keys:
        ax.plot(histories[k]['val_loss'], '-', label=f"{k}", linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val Loss')
    ax.set_title('Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot val dice
    ax = axes[1, 0]
    for k in pinn_keys:
        ax.plot(histories[k]['val_dice'], '-', label=f"{k} (PINN)", linewidth=2)
    for k in nopinn_keys:
        ax.plot(histories[k]['val_dice'], '-', label=f"{k}", linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val Dice')
    ax.set_title('Validation Dice Score')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Hide unused subplot
    axes[1, 1].axis('off')
    
    plt.suptitle('Ablation Study: PINN TV Loss vs Baseline', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

if model_histories:
    plot_ablation_curves(model_histories)
else:
    print("Belum ada history training. Jalankan sel training terlebih dahulu.")

---
### 12. PINN TV Loss - Hyperparameter Tuning

Bobot TV Loss (`tv_weight`) adalah hiperparameter penting. Cell ini memungkinkan eksperimen dengan berbagai nilai tv_weight untuk menemukan yang optimal.

In [ ]:
# Eksperimen dengan berbagai tv_weight
tv_weights = [0.001, 0.005, 0.01, 0.05, 0.1]
tuning_results = {}

print("=" * 60)
print("   Hyperparameter Tuning: PINN TV Weight")
print("=" * 60)

for tv_w in tv_weights:
    print(f"\n>>> Testing tv_weight = {tv_w}")
    model = UNet().cuda()
    
    # Train with quick 5 epochs for tuning
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = PINN_BCEDiceTverskyTVLoss(bce_weight=0.5, tv_weight=tv_w).cuda()
    
    for epoch in range(5):
        model.train()
        for images, masks, _ in train_loader:
            images, masks = images.cuda(), masks.cuda()
            optimizer.zero_grad()
            seg_out = model(images)
            loss = criterion(seg_out, masks)
            loss.backward()
            optimizer.step()
    
    # Quick eval
    model.eval()
    val_dice = 0
    with torch.no_grad():
        for images, masks, _ in val_loader:
            images, masks = images.cuda(), masks.cuda()
            seg_out = model(images)
            val_dice += dice_coeff(seg_out, masks)
    
    avg_dice = val_dice / len(val_loader)
    tuning_results[tv_w] = avg_dice
    print(f"   Val Dice setelah 5 epoch: {avg_dice:.4f}")

print("\n" + "=" * 60)
print(f"{'tv_weight':<12} | {'Val Dice':<10}")
print("-" * 25)
best_tv = max(tuning_results, key=tuning_results.get)
for tv_w, dice in tuning_results.items():
    marker = " <<< BEST" if tv_w == best_tv else ""
    print(f"{tv_w:<12} | {dice:<10.4f}{marker}")
print(f"\n>>> Recommended tv_weight = {best_tv}")
print("=" * 60)

---
### 13. Ringkasan & Kesimpulan

**Apa yang baru di V7?**

1. **Physics-Informed Neural Network (PINN) Integration**
   - Menambahkan **Total Variation (TV) Loss** sebagai regularisasi fisika
   - TV Loss meminimalkan gradien spasial → mendorong segmentasi yang halus dan kontinu
   - Mengurangi false positive akibat class imbalance

2. **Loss Function Baru: PINN_BCEDiceTverskyTVLoss**
   - Weighted Focal Loss → menangani imbalance piksel
   - Tversky Loss (α=0.3, β=0.7) → menangani imbalance region, fokus pada false negatives
   - **TV Loss** (PINN) → regularisasi fisika untuk segmentasi yang smooth

3. **Ablation Study**
   - Setiap model dilatih dengan dan tanpa PINN untuk perbandingan
   - Visualisasi side-by-side menunjukkan perbedaan kualitas segmentasi

4. **Hyperparameter Tuning**
   - Eksperimen dengan berbagai bobot TV Loss (tv_weight)

**Referensi:**
- Ding et al., "PINN-EMFNet: PINN-based and Enhanced Multi-Scale Feature Fusion Network for Breast Ultrasound Images Segmentation", 2024
- Raissi et al., "Physics-Informed Neural Networks: A Deep Learning Framework for Solving Forward and Inverse Problems Involving Nonlinear Partial Differential Equations", 2019
- Banerjee et al., "PINNs for Medical Image Analysis: A Survey", 2024